# STM–BERTopic triangulation (Appendix 6, Test 3 / Panel C)

Companion code for: "Measuring Organizational Constructs in Large-Scale Text:
An interpretive framework combining topic modeling and BERT", *Organization Studies*.

Independent, embedding-based topic model used to check whether the STM topic
structure is method-specific. Produces a 30-topic BERTopic solution and its top
words, which are manually matched to the STM topics in Panel C. Input:
`texts_for_bertopic_base.csv`, the STM-cleaned corpus exported by the R script.

In [ ]:
# --- Install (uncomment in a fresh environment / Colab) ---
# !pip install bertopic sentence-transformers scikit-learn pandas numpy

# --- Imports ---
import re
from pathlib import Path

import pandas as pd
import numpy as np

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer


from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

In [ ]:
# --- Input: STM-cleaned corpus exported from the R script ---
INPUT_CSV = Path("texts_for_bertopic_base.csv")

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"CSV file not found: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV, encoding="utf-8")
if "text" not in df.columns:
    raise ValueError("CSV must contain a 'text' column.")

texts = df["text"].fillna("").astype(str).tolist()
print(f"Loaded {len(texts)} documents.") 


# --- Light preprocessing for BERTopic (text is already STM-cleaned) ---
def simple_preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)  # Remove non-letters
    tokens = text.split()
    tokens = [w for w in tokens if w not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)

texts_cleaned = [simple_preprocess(doc) for doc in texts]
print(f"Cleaned {len(texts_cleaned)} documents.")


In [ ]:
# --- Load embedding model & generate embeddings from cleaned text ---
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# --- Generate embeddings from cleaned text ---
embeddings = embedder.encode(
    texts_cleaned,
    show_progress_bar=True,
    normalize_embeddings=True  
)

print(f"Generated {len(embeddings)} embeddings.")  

In [ ]:
# Cluster embeddings into exactly 30 topics with k-means (in place of BERTopic's
# default density-based clustering) for a direct 30-vs-30 comparison with STM.
# A seeded k-means produces label assignments that guide a semi-supervised
# BERTopic run whose clustering is fixed to K = 30.

K = 30
kmeans = KMeans(n_clusters=K, random_state=42)
predicted_labels = kmeans.fit_predict(embeddings)

# --- Custom vectorizer for topic representation ---
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95
)

# --- BERTopic with k-means as the cluster model (predicted_labels guide it via y) ---
topic_model = BERTopic(
    hdbscan_model=KMeans(n_clusters=K),
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

# --- Fit the model using pre-calculated topics via the 'y' parameter ---
topics, probs = topic_model.fit_transform(
    documents=texts_cleaned,
    embeddings=embeddings,
    y=predicted_labels
)

print(f"Model trained with exactly {K} topics (manual KMeans).")



In [ ]:
# Prepare topic assignments and inspect

# Topic summary (topic id, count, top words)
topic_info_df = topic_model.get_topic_info()

# Cleaner topic names: "Topic 0: word1, word2, ..."  (KMeans has no -1 outlier)
topic_id_to_name = {
    row.Topic: f"Topic {row.Topic}: {row.Name.replace('_', ', ')}"
    for _, row in topic_info_df.iterrows()
    if row.Topic != -1
}

# Document-topic assignments 
final_assignments_df = pd.DataFrame({"text": texts_cleaned, "topic_id": topics})
final_assignments_df["topic_name"] = (
    final_assignments_df["topic_id"].map(topic_id_to_name).fillna("Outlier Topic")
)

# --- Inspect ---
n_topics = (topic_info_df["Topic"] != -1).sum()
print(f"--- Topic Summary (KMeans): {n_topics} topics ---")
display(topic_info_df.head(10))

print("\n--- Final Document-Topic Assignments ---")
display(final_assignments_df.head(10))

print("\n--- Document Count Per Topic ---")
display(final_assignments_df["topic_name"].value_counts().to_frame())

In [ ]:
# --- Save outputs ---
# NOTE: bertopic_document.csv contains review text (document-level assignments)
# and is for local use only; bertopic_topic.csv (topic-word lists) is shareable.

topic_info_df.to_csv("bertopic_topic.csv", index=False)
print("Saved: bertopic_topic.csv")

final_assignments_df.to_csv("bertopic_document.csv", index=False)  
print("Saved: bertopic_document.csv")

# Note: the clustering step is stochastic, so results may differ
# slightly from the specific run reported in the paper.

In [ ]:
import sys
!{sys.executable} -m pip freeze > bertopic_requirements.txt
print("Saved: bertopic_requirements.txt")